In [1]:
pip install torch torchvision torch-geometric huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.9 MB/s eta 0:00:00


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
api = HfApi()

In [ ]:
repo_id = "archi829/bottleneck-oracle-graphs"
api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

In [2]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity
import torch_geometric
from torch_geometric.data import HeteroData
import networkx as nx
import json
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/graphs", exist_ok=True)

In [3]:
configs = [
    {"name": "tiny",   "d_model": 64,  "nhead": 2, "num_layers": 1},
    {"name": "small",  "d_model": 128, "nhead": 4, "num_layers": 2},
    {"name": "medium", "d_model": 256, "nhead": 8, "num_layers": 4},
]

In [ ]:
def generate_trace(config, trace_id):
    """
    Generate a trace with stage-level coarsening.
    
    Groups operators by transformer stage (e.g., one transformer layer = one node)
    to reduce graph size from millions of operators to hundreds of stages.
    """
    model = nn.Transformer(
        d_model=config["d_model"],
        nhead=config["nhead"],
        num_encoder_layers=config["num_layers"],
        num_decoder_layers=config["num_layers"]
    )

    src = torch.rand(10, 32, config["d_model"])
    tgt = torch.rand(10, 32, config["d_model"])

    # Enable stack tracing for stage grouping
    with profile(
        activities=[ProfilerActivity.CPU], 
        record_shapes=True,
        with_stack=True  # Required for stage-level grouping
    ) as prof:
        with record_function("model_forward"):
            output = model(src, tgt)

    filepath = f"data/raw/trace_{config['name']}_{trace_id}.json"
    prof.export_chrome_trace(filepath)
    return filepath

# test with one trace first
path = generate_trace(configs[0], 0)
print(f"✅ Test trace saved: {path}")

In [5]:
def parse_trace(filepath):
    with open(filepath) as f:
        trace = json.load(f)

    ops = [
        e for e in trace["traceEvents"]
        if e.get("cat") == "cpu_op" and "dur" in e
    ]
    ops = sorted(ops, key=lambda x: x["ts"])

    G = nx.DiGraph()
    for i, op in enumerate(ops):
        node_type = "network" if "allreduce" in op["name"].lower() or "comm" in op["name"].lower() else "compute"
        G.add_node(i,
            name=op["name"],
            type=node_type,
            duration_ms=op["dur"] / 1000,
            timestamp=op["ts"]
        )
        if i > 0:
            G.add_edge(i-1, i)
    return G

# test
G = parse_trace(path)
print(f"✅ Parser working: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

✅ Parser working: 574 nodes, 573 edges


In [ ]:
import re
from collections import defaultdict

def extract_module_from_stack(event):
    """
    Extract module name and layer index from stack trace.
    
    Examples:
        "torch.nn.modules.transformer.TransformerEncoder.forward" → ("TransformerEncoder", 0)
        "torch.nn.modules.linear.Linear.forward" → ("Linear", -1)
    """
    if "stack" not in event or not event["stack"]:
        return ("unknown", -1)
    
    # Get the topmost frame (most recent call)
    top_frame = event["stack"][-1] if event["stack"] else {}
    func_name = top_frame.get("name", "")
    
    # Pattern match for transformer components
    if "TransformerEncoder" in func_name:
        # Extract layer index if present
        match = re.search(r'layer(\d+)', func_name)
        layer_idx = int(match.group(1)) if match else 0
        return ("TransformerEncoder", layer_idx)
    elif "TransformerDecoder" in func_name:
        match = re.search(r'layer(\d+)', func_name)
        layer_idx = int(match.group(1)) if match else 0
        return ("TransformerDecoder", layer_idx)
    elif "self_attn" in func_name or "multihead" in func_name.lower():
        return ("SelfAttention", -1)
    elif "linear" in func_name.lower():
        return ("Linear", -1)
    else:
        return ("Other", -1)

def group_ops_by_stage(ops):
    """
    Group operators by stage (module + layer combination).
    
    Returns:
        stages: List of dicts with keys:
            - name: stage name
            - layer_idx: layer index
            - ops: list of original operator indices
            - duration_ms: sum of all operator durations
            - timestamp: earliest start time
            - end_time: latest end time
    """
    stage_groups = defaultdict(lambda: {
        "ops": [],
        "durations": [],
        "timestamps": [],
        "end_times": [],
        "names": []
    })
    
    for i, op in enumerate(ops):
        module, layer_idx = extract_module_from_stack(op)
        key = (module, layer_idx)
        
        stage_groups[key]["ops"].append(i)
        stage_groups[key]["durations"].append(op["dur"] / 1000.0)  # Convert to ms
        stage_groups[key]["timestamps"].append(op["ts"])
        stage_groups[key]["end_times"].append(op["ts"] + op["dur"])
        stage_groups[key]["names"].append(op.get("name", f"op_{i}"))
    
    # Convert to list of stage dicts
    stages = []
    for (module, layer_idx), group in sorted(stage_groups.items()):
        stages.append({
            "name": f"{module}_{layer_idx}" if layer_idx >= 0 else module,
            "module": module,
            "layer_idx": layer_idx,
            "ops": group["ops"],
            "duration_ms": sum(group["durations"]),
            "timestamp": min(group["timestamps"]),
            "end_time": max(group["end_times"]),
            "op_count": len(group["ops"])
        })
    
    return stages

print("✅ Stage grouping functions defined")

In [ ]:
def parse_trace_with_stages(filepath):
    """
    Parse trace and group by stages for coarser granularity.
    
    This reduces graph size by ~100-1000x while preserving timing dependencies.
    """
    with open(filepath) as f:
        trace = json.load(f)

    ops = [
        e for e in trace["traceEvents"]
        if e.get("cat") == "cpu_op" and "dur" in e
    ]
    ops = sorted(ops, key=lambda x: x["ts"])
    
    # Group by stage
    stages = group_ops_by_stage(ops)
    
    G = nx.DiGraph()
    for i, stage in enumerate(stages):
        node_type = "network" if "allreduce" in stage["name"].lower() or "comm" in stage["name"].lower() else "compute"
        G.add_node(i,
            name=stage["name"],
            type=node_type,
            duration_ms=stage["duration_ms"],
            timestamp=stage["timestamp"],
            end_time=stage["end_time"],
            op_count=stage["op_count"],
            module=stage["module"],
            layer_idx=stage["layer_idx"]
        )
        if i > 0:
            G.add_edge(i-1, i)
    
    return G, stages

# Test stage-level parsing
G_stages, stages = parse_trace_with_stages(path)
print(f"✅ Stage-level parsing: {G_stages.number_of_nodes()} stage nodes (vs {len(stages)} original ops)")
print(f"✅ Compression ratio: {len(stages) / max(G_stages.number_of_nodes(), 1):.1f}x")
print(f"✅ Sample stages: {[s['name'] for s in stages[:5]]}")

In [6]:
def find_critical_path(G):
    for u, v in G.edges():
        G[u][v]['weight'] = G.nodes[u]['duration_ms']
    critical_path = nx.dag_longest_path(G, weight='weight')
    critical_path_length = nx.dag_longest_path_length(G, weight='weight')
    return critical_path, critical_path_length

def add_features(G, critical_path):
    total_time = sum(G.nodes[n]['duration_ms'] for n in G.nodes())
    max_dur = max(G.nodes[n]['duration_ms'] for n in G.nodes())
    for n in G.nodes():
        duration = G.nodes[n]['duration_ms']
        G.nodes[n]['compute_ratio'] = duration / total_time
        G.nodes[n]['is_critical'] = 1 if n in critical_path else 0
        G.nodes[n]['norm_duration'] = duration / max_dur
    return G

# test
critical_path, total_time = find_critical_path(G)
G = add_features(G, critical_path)
print(f"✅ Features added. Total time: {total_time:.2f}ms")

✅ Features added. Total time: 682.31ms


In [8]:
def to_pyg(G, total_time_ms):
    compute_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'compute']
    network_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'network']

    # map original node id → local index (separately for each type)
    compute_idx = {n: i for i, n in enumerate(compute_nodes)}
    network_idx = {n: i for i, n in enumerate(network_nodes)}

    data = HeteroData()

    # ── compute node features (Story 5: proxy features, non-zero, normalized) ──
    data['compute'].x = torch.tensor([
        [
            G.nodes[n]['duration_ms'],        # raw duration
            G.nodes[n]['compute_ratio'],       # compute ratio (duration / total)
            G.nodes[n]['norm_duration'],       # norm duration (duration / max)
            # comm/compute ratio: 0.0 for pure compute nodes (no comm)
            0.0
        ]
        for n in compute_nodes
    ], dtype=torch.float)

    # ── node-level label: is_critical (move out of features → y) ──
    data['compute'].y = torch.tensor(
        [G.nodes[n]['is_critical'] for n in compute_nodes],
        dtype=torch.float
    )

    # ── network node features ──
    if network_nodes:
        data['network'].x = torch.tensor([
            [
                G.nodes[n]['duration_ms'],
                G.nodes[n]['compute_ratio'],
                G.nodes[n]['norm_duration'],
                1.0   # comm/compute ratio = 1.0 for network nodes
            ]
            for n in network_nodes
        ], dtype=torch.float)
        data['network'].y = torch.tensor(
            [G.nodes[n]['is_critical'] for n in network_nodes],
            dtype=torch.float
        )
    else:
        data['network'].x = torch.zeros((0, 4))
        data['network'].y = torch.zeros((0,))

    # ── edges: compute → compute (Story 3: directed, no bidirectional leakage) ──
    cc_src, cc_dst = [], []
    cn_src, cn_dst = [], []
    nc_src, nc_dst = [], []

    for u, v in G.edges():
        u_type = G.nodes[u]['type']
        v_type = G.nodes[v]['type']
        if u_type == 'compute' and v_type == 'compute':
            cc_src.append(compute_idx[u])
            cc_dst.append(compute_idx[v])
        elif u_type == 'compute' and v_type == 'network':
            cn_src.append(compute_idx[u])
            cn_dst.append(network_idx[v])
        elif u_type == 'network' and v_type == 'compute':
            nc_src.append(network_idx[u])
            nc_dst.append(compute_idx[v])

    if cc_src:
        data['compute', 'depends_on', 'compute'].edge_index = torch.tensor(
            [cc_src, cc_dst], dtype=torch.long)
    if cn_src:
        data['compute', 'sends_to', 'network'].edge_index = torch.tensor(
            [cn_src, cn_dst], dtype=torch.long)
    if nc_src:
        data['network', 'feeds', 'compute'].edge_index = torch.tensor(
            [nc_src, nc_dst], dtype=torch.long)

    # ── graph-level label: total step time ──
    data.y = torch.tensor([total_time_ms], dtype=torch.float)

    return data

data = to_pyg(G, total_time_ms=total_time)
print(f"✅ PyG working: compute={data['compute'].x.shape}")

✅ PyG working: compute=torch.Size([574, 4])


In [ ]:
# all_graphs = []

# for config in configs:
#     for trace_id in range(167):  # ~500 total across 3 configs
#         filepath = generate_trace(config, trace_id)
#         G = parse_trace(filepath)
#         critical_path, total_time = find_critical_path(G)
#         G = add_features(G, critical_path)
#         data = to_pyg(G, total_time_ms=total_time)

#         # save pyg graph
#         torch.save(data, f"data/graphs/graph_{config['name']}_{trace_id}.pt")
#         all_graphs.append(data)

#         if trace_id % 50 == 0:
#             print(f"✅ {config['name']} — {trace_id}/167 done")

# print(f"\n🎉 Total graphs generated: {len(all_graphs)}")

✅ tiny — 0/167 done
✅ tiny — 50/167 done
✅ tiny — 100/167 done
✅ tiny — 150/167 done


In [ ]:
import shutil

all_graphs = []
pending_files = []

def batch_upload_and_clear(pending_files):
    """Upload only pending files to HF, then clear the staging area."""
    staging_dir = "data/graphs_staging"
    os.makedirs(staging_dir, exist_ok=True)

    for f in pending_files:
        shutil.copy(f, os.path.join(staging_dir, os.path.basename(f)))

    api.upload_folder(
        folder_path=staging_dir,
        repo_id="archi829/bottleneck-oracle-graphs",
        repo_type="dataset"
    )

    shutil.rmtree(staging_dir)
    os.makedirs(staging_dir, exist_ok=True)
    return []

for config in configs:
    for trace_id in range(167):
        filepath = generate_trace(config, trace_id)
        G = parse_trace(filepath)
        critical_path, total_time = find_critical_path(G)
        G = add_features(G, critical_path)
        data = to_pyg(G, total_time_ms=total_time)

        local_path = f"data/graphs/graph_{config['name']}_{trace_id}.pt"
        torch.save(data, local_path)
        all_graphs.append(data)
        pending_files.append(local_path)

        if len(pending_files) == 50:
            pending_files = batch_upload_and_clear(pending_files)
            print(f"✅ {config['name']} — {trace_id+1}/167 done + batch uploaded")

# upload whatever's left
if pending_files:
    count = len(pending_files)
    batch_upload_and_clear(pending_files)
    pending_files = []
    print(f"✅ Final batch uploaded ({count} files)")

print(f"\n🎉 Done! {len(all_graphs)} graphs generated and uploaded")

In [ ]:
g = torch.load("data/graphs/graph_medium_0.pt", weights_only=False)
assert g['compute'].x.shape[1] == 4
assert (g['compute'].x[:, :3] > 0).all()   # non-zero proxy features
assert g['compute'].y is not None            # node labels exist
assert g.y is not None                       # graph label exists
assert len(g.edge_types) > 0                 # edges exist
print("✅ All Story 3/4/5 acceptance criteria passed")